In [85]:
%run Distribution_of_constant_symbol.ipynb
%run Geom_Prolongation.ipynb
from sympy import IndexedBase

In [ ]:
def Flat_distr(g):
    """INPUTS:
    * 'g' - a T_symbol object"""
    C=cochain_complex(g)
    return Distr_of_constant_symbol(g,C.elt({}))

In [86]:
def Free_distr(g):
    """INPUTS:
    * 'g' - a T_symbol object"""
    K=IndexedBase('K')
    C=cochain_complex(g)
    c=C.elt({})

    for i in range(len(g.basis)):
        if g.basis[i].wght<0:
            for j in range(i+1,len(g.basis)):
                for k in range(len(g.basis)):
                    if g.basis[k].wght<0 and g.basis[k].wght>g.basis[i].wght+g.basis[j].wght:
                        c=c+C.elt_from_cd({(str(g.basis[i]),str(g.basis[j]),str(g.basis[k])):K[i,j,k]})
    return Distr_of_constant_symbol(g,c)

In [ ]:
def J_subs_dict():
    g=Symp_symb(7)
    D=Free_distr(g)
    K=IndexedBase('K')
    J_subs={}
    wght_1_list=[]
    wght_2_list=[]
    wght_3_list=[]
    for i in range(3,len(g.basis)):
        for j in range(i+1,len(g.basis)):
            for k in range(j+1,len(g.basis)):
                for l in range(3,len(g.basis)):
                    if g.basis[i].wght+g.basis[j].wght+g.basis[k].wght-g.basis[l].wght==-1:
                        wght_1_list.append((i,j,k,l))
                    if g.basis[i].wght+g.basis[j].wght+g.basis[k].wght-g.basis[l].wght==-2:
                        wght_2_list.append((i,j,k,l))
                    if g.basis[i].wght+g.basis[j].wght+g.basis[k].wght-g.basis[l].wght==-3:
                        wght_3_list.append((i,j,k,l))

    # wght 1
    K_list=[K[4,5,5],K[4,6,6],K[4,7,7],K[4,8,8],K[3,10,10],K[5,6,7],K[5,7,8],K[3,5,5],K[3,6,6],K[5,8,9]]
    JI_ind_list=[A for A in wght_1_list if A not in [(4,5,6,8),(4,5,7,9),(4,6,7,10)]]
    for i in range(len(K_list)):
        s=solve(D.Jacobi_id(*JI_ind_list[i]).subs(J_subs),K_list[i])
        if len(s)==0: print(JI_ind_list[i], 'has no solutions for ',K_list[i])
        back_substitute(J_subs,K_list[i],s[0])
        J_subs[K_list[i]]=s[0]

    # wght 2
    # Add in derivatives
    for A in list(J_subs.keys()):
        # To do: figure out how to make sure all this is in normal form
        t3=D.normalize_der(D.abn_ind_der(J_subs[A],3))
        J_subs[D.abn_ind_der(A,3)]=t3
        back_substitute(J_subs,D.abn_ind_der(A,3),t3)
        
        t4=D.normalize_der(D.abn_ind_der(J_subs[A],4))
        J_subs[D.abn_ind_der(A,4)]=t4
        back_substitute(J_subs,D.abn_ind_der(A,4),t4)

    K_list=[K[3,5,3],K[4,6,5],K[4,7,6],K[4,8,7],K[4,9,8],K[5,10,10],K[5,7,7],K[5,8,8],K[5,9,9],K[6, 9, 10],
        K[6, 8, 9],K[7, 8, 10],K[4, 5, 3],0,K[4, 10, 9],K[6, 7, 8],0]
    JI_ind_list=[A for A in wght_2_list if A not in []]
    for i in range(len(K_list)):
        if K_list[i]!=0:
            t=D.Jacobi_id(*JI_ind_list[i]).subs(J_subs)
            s=solve(t,K_list[i])
            if len(s)==0: print('\n',JI_ind_list[i],'has no solutions for ',K_list[i])
            sol=s[0]
            back_substitute(J_subs,K_list[i],sol)
            J_subs[K_list[i]]=sol

    # wght 3
    # Add in derivatives
    for A in list(J_subs.keys()):
        w=-g.basis[A.indices[0]].wght-g.basis[A.indices[1]].wght+g.basis[A.indices[2]].wght
        for i in range(3,len(A.indices)):
            w+=-g.basis[A.indices[i]].wght
        if w==2:
            t3=D.normalize_der(D.abn_ind_der(J_subs[A],3))
            J_subs[D.abn_ind_der(A,3)]=t3
            back_substitute(J_subs,D.abn_ind_der(A,3),t3)
            t4=D.normalize_der(D.abn_ind_der(J_subs[A],4))
            J_subs[D.abn_ind_der(A,4)]=t4
            back_substitute(J_subs,D.abn_ind_der(A,4),t4)
        if w==1:
            t5=D.normalize_der(D.abn_ind_der(J_subs[A],5))
            J_subs[D.abn_ind_der(A,5)]=t5
            back_substitute(J_subs,D.abn_ind_der(A,5),t5)

    K_list=[K[5,6,6,4],K[4,6,4],K[5,6,5],K[5,7,6],K[5,8,7],K[5,9,8],K[5,10,9],K[4,8,6],K[6,7,7],K[6,8,8],K[6,9,9],K[6,10,10],
        K[3,10,8],K[7,8,9],K[7,9,10],K[3,8,6],K[4,5,4,4],0,K[3,10,9,4],K[6,7,9,4,3],0,0,0,0,0,0,0]
    JI_ind_list=[A for A in wght_3_list if A not in []]
    for i in range(len(K_list)):
        if K_list[i]!=0:
            t=D.Jacobi_id(*JI_ind_list[i]).subs(J_subs)
            s=solve(t,K_list[i])
            if len(s)==0: print(JI_ind_list[i],'has no solutions for',K_list[i])
            sol=s[0]
            back_substitute(J_subs,K_list[i],sol)
            J_subs[K_list[i]]=sol.subs(J_subs)
    return J_subs

In [ ]:
def Prenorm_JS_dict():
    # Up through weight 4 for the moment

    g=Symp_symb(7)
    D=Free_distr(g)
    K=IndexedBase('K')
    result={}
    for k in range(4,11-2):
        for k1 in range(3,k+1):
            result[K[3,k,k1]]=0
            for i in range(3,11):
                result[K[3,k,k1,i]]=0
                result[K[3,k,k1,i]]=0
                for j in range(3,11):
                    result[K[3,k,k1,i,j]]=0

    # Canonical section
    result[K[3,9,9]]=0
    # Projective parametrization
    result[K[3,9,8]]=0
    for k in range(3,11-1):
        result[K[4,9,k]]=0
        result[K[3,9,9,k]]=0
        result[K[3,9,8,k]]=0
        for k1 in range(3, 11-1):
            result[K[4,9,k,k1]]=0
            result[K[3,9,9,k,k1]]=0
            result[K[3,9,8,k,k1]]=0

    wght_1_list=[]
    wght_2_list=[]
    wght_3_list=[]
    wght_4_list=[]
    wght_5_list=[]
    wght_6_list=[]
    wght_7_list=[]

    for i in range(3,len(g.basis)):
        for j in range(i+1,len(g.basis)):
            for k in range(j+1,len(g.basis)):
                for l in range(3,len(g.basis)):
                    w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[l].wght
                    if w in range(1,7):
                        wl=[None,wght_1_list,wght_2_list,wght_3_list,wght_4_list,wght_5_list,wght_6_list][w]
                        wl.append((i,j,k,l))
    # wght 1
    K_list=[K[4,5,5],K[4,6,6],K[4,7,7],K[4,8,8],K[3,10,10],K[5,6,7],K[5,7,8],K[6,8,10],K[5,9,10],K[6,7,9]]
    JI_ind_list=[A for A in wght_1_list if A not in [(4,5,6,8),(4,5,7,9),(4,6,7,10)]]
    for i in range(len(K_list)):
        s=solve(D.Jacobi_id(*JI_ind_list[i]).subs(result),K_list[i])
        if len(s)==0: print(JI_ind_list[i], 'has no solutions for ',K_list[i])
        back_substitute(result,K_list[i],s[0])
        result[K_list[i]]=s[0]

    # wght 2
    # Add in derivatives
    for A in list(result.keys()):
        for dir in range(3,len(g.basis)):
            if wght_of_ind(A,g)+g.basis[dir].wght==-2:
                add_subs_der(A,dir,result,D)

    # wght 2
    K_list=[K[4,5,4],K[4,6,5],K[4,7,6],K[4,8,7],K[5,9,9],K[5,10,10],K[5,7,7],K[5,8,8],K[5,6,6],K[6,9,10],K[6,8,9],
            K[7,8,10],K[4,5,3],K[4,10,9],K[6,7,8]]
    JI_ind_list=[A for A in wght_2_list if A not in [(4,5,7,8),(4,6,8,10),(4,6,7,9),(5,6,7,10)]]
    for i in range(len(K_list)):
        s=solve(D.Jacobi_id(*JI_ind_list[i]).subs(result),K_list[i])
        if len(s)==0: print(JI_ind_list[i], 'has no solutions for ',K_list[i])
        back_substitute(result,K_list[i],s[0])
        result[K_list[i]]=s[0]

    # # wght 3
    # Add in derivatives
    for A in list(result.keys()):
        for dir in range(3,len(g.basis)):
            if wght_of_ind(A,g)+g.basis[dir].wght==-3:
                add_subs_der(A,dir,result,D)

    K_list=[K[4,6,3],K[4,6,4],K[4,7,5],K[4,8,6],K[5,6,5],K[5,9,8],K[5,10,9],K[5,8,7],K[5,7,6],K[6,7,7],K[6,9,9],
            K[6,10,10],K[6,8,8],K[7,8,9],K[7,9,10],K[3,9,7],K[3,10,9,4],K[4,10,8],K[4,10,10,4,3]]
    JI_ind_list=[A for A in wght_3_list if A not in [(4,5,7,7),(4,6,7,8),(4,5,10,10),
                                                    (4,6,8,9),(4,6,9,10),(4,7,8,10),(5,6,7,9),(5,6,8,10)]]
    for i in range(len(K_list)):
        s=solve(D.Jacobi_id(*JI_ind_list[i]).subs(result),K_list[i])
        if len(s)==0: print(JI_ind_list[i], 'has no solutions for ',K_list[i])
        back_substitute(result,K_list[i],s[0])
        result[K_list[i]]=s[0]

    # # wght 4
    # Add in derivatives
    for A in list(result.keys()):
        for dir in range(3,len(g.basis)):
            if wght_of_ind(A,g)+g.basis[dir].wght==-4:
                add_subs_der(A,dir,result,D)

    K_list=[K[5,6,3],K[5,6,4],K[5,7,5],K[5,8,6],K[5,9,7],K[4,10,7],K[4,7,4],K[6,7,6],K[4,8,5],K[6,9,8],K[6,10,9],
            K[6,8,7],K[7,8,8],K[7,9,9],K[7,10,10],K[3,10,7],K[8,9,10],K[3,10,8,4],K[5,8,9,4,3,3],K[4,7,3],K[5,8,9,4,4,3]]

    JI_ind_list=[A for A in wght_4_list if A not in [(4,5,7,6),(4,6,7,7),(4,6,8,8),(4,6,9,9),(4,6,10,10),(4,7,8,9),(4,7,9,10),
                                                    (5,6,7,8),(5,6,8,9),(5,6,9,10),(5,7,8,10)]]

    for i in range(len(K_list)):
        s=solve(D.Jacobi_id(*JI_ind_list[i]).subs(result),K_list[i])
        if len(s)==0: print(JI_ind_list[i], 'has no solutions for ',K_list[i])
        back_substitute(result,K_list[i],s[0])
        result[K_list[i]]=s[0]
    return result

In [ ]:
# To do: Think about prenormalization and construct a prenormalized distribution

Norm_subs={}
for k in range(4,11-2):
    for k1 in range(3,k+1):
        Norm_subs[K[3,k,k1]]=0
        for i in range(3,11):
            Norm_subs[K[3,k,k1,i]]=0
            Norm_subs[K[3,k,k1,i]]=0
            for j in range(3,11):
                Norm_subs[K[3,k,k1,i,j]]=0

# Canonical section
Norm_subs[K[3,9,9]]=0
# Projective parametrization
Norm_subs[K[3,8,8]]=0
# Self-dual curve
Norm_subs[K[3,7,7]]=0
for k in range(3,11-1):
    Norm_subs[K[4,9,k]]=0
    Norm_subs[K[3,9,9,k]]=0
    Norm_subs[K[3,8,8,k]]=0
    Norm_subs[K[3,7,7,k]]=0
    for k1 in range(3, 11-1):
        Norm_subs[K[4,9,k,k1]]=0
        Norm_subs[K[3,9,9,k,k1]]=0
        Norm_subs[K[3,8,8,k,k1]]=0
        Norm_subs[K[3,7,7,k,k1]]=0

In [138]:
def Free_distr_Jacobi_7(g):
    """Returns a Free_distr(g), where g=Symp_symb(7) with the Jacobi Identity applied 
    in the curvature up to weight 3. Note: More applications of the Jacobi Identity may be needed
    """
    r=Free_distr(g)
    r.curv=r.curv.subs(J_subs_dict())
    return r

In [1]:
def Prenorm_distr_Jacobi_7(g):
    """Returns a Free_distr(g), where g=Symp_symb(7) with the Jacobi Identity applied 
    in the curvature up to weight 3. Note: More applications of the Jacobi Identity may be needed
    """
    r=Free_distr(g)
    r.curv=r.curv.subs(Prenorm_JS_dict())
    return r